In [ ]:
%pip install tensorflow

  Using cached tensorflow-2.21.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (4.4 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached protobuf-7.36.2-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)


In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 250

(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

word_index = imdb.get_word_index()
index_word = {index + 3: word for word, index in word_index.items()}
index_word[0] = "<PAD>"
index_word[1] = "<START>"
index_word[2] = "<UNK>"
index_word[3] = "<UNUSED>"

def decode_review(sequence):
    return " ".join(index_word.get(index, "<UNK>") for index in sequence if index > 3)

train_texts = [decode_review(seq) for seq in X_train_raw]
test_texts = [decode_review(seq) for seq in X_test_raw]

print("Training samples:", len(train_texts))
print("Testing samples:", len(test_texts))
print("Vocabulary size:", VOCAB_SIZE)
print("Sample review:", train_texts[0][:500])
print("Sample label:", "Positive" if y_train[0] == 1 else "Negative")

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_train_texts = [clean_text(text) for text in train_texts]
clean_test_texts = [clean_text(text) for text in test_texts]

print(clean_train_texts[0][:500])

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_text(text):
    tokens = text.split()
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [stemmer.stem(token) for token in tokens]
    return " ".join(tokens)

processed_train_texts = [preprocess_text(text) for text in clean_train_texts]
processed_test_texts = [preprocess_text(text) for text in clean_test_texts]

print(processed_train_texts[0][:500])

In [ ]:
count_vectorizer = CountVectorizer(max_features=10000)
X_train_bow = count_vectorizer.fit_transform(processed_train_texts)
X_test_bow = count_vectorizer.transform(processed_test_texts)

tfidf_vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf_vectorizer.fit_transform(processed_train_texts)
X_test_tfidf = tfidf_vectorizer.transform(processed_test_texts)

print("BoW training shape:", X_train_bow.shape)
print("TF-IDF training shape:", X_train_tfidf.shape)

In [ ]:
X_train = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding="post", truncating="post")
X_test = pad_sequences(X_test_raw, maxlen=MAX_LEN, padding="post", truncating="post")

X_train.shape, X_test.shape

In [ ]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    LSTM(64, dropout=0.2, recurrent_dropout=0.0),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    np.array(y_train),
    epochs=8,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1]
})

display(metrics_df)
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    "Actual": y_test[:30],
    "Predicted": y_pred[:30]
})

comparison_df["Actual Sentiment"] = comparison_df["Actual"].map({0: "Negative", 1: "Positive"})
comparison_df["Predicted Sentiment"] = comparison_df["Predicted"].map({0: "Negative", 1: "Positive"})

display(comparison_df[["Actual Sentiment", "Predicted Sentiment"]])

plt.figure(figsize=(12, 5))
positions = np.arange(len(comparison_df))
plt.scatter(positions, comparison_df["Actual"], label="Actual", marker="o")
plt.scatter(positions, comparison_df["Predicted"], label="Predicted", marker="x")
plt.yticks([0, 1], ["Negative", "Positive"])
plt.xlabel("Review Index")
plt.ylabel("Sentiment")
plt.title("Actual vs Predicted Sentiment (First 30 Test Reviews)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    sequence = []
    for token in cleaned.split():
        index = word_index.get(token)
        if index is None or index + 3 >= VOCAB_SIZE:
            sequence.append(2)
        else:
            sequence.append(index + 3)
    sequence = pad_sequences([sequence], maxlen=MAX_LEN, padding="post", truncating="post")
    probability = float(model.predict(sequence, verbose=0)[0][0])
    label = "Positive" if probability >= 0.5 else "Negative"
    return label, probability

sample_reviews = [
    "This movie was amazing, excellent, and entertaining. I loved every minute.",
    "The movie was boring, disappointing, and a complete waste of time.",
    "The acting was good but the story was not very interesting."
]

for review in sample_reviews:
    sentiment, probability = predict_sentiment(review)
    print("Review:", review)
    print("Prediction:", sentiment)
    print("Positive probability:", round(probability, 4))
    print()

In [ ]:
model.save("imdb_lstm_sentiment_model.keras")
print("Model saved successfully.")

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test Loss:", round(float(test_loss), 4))
print("Test Accuracy:", round(float(test_accuracy), 4))

In [ ]:
results = pd.DataFrame({
    "Model": ["LSTM with Embeddings"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1 Score": [f1]
})
display(results)